# Investigating control/PUA characters in `chunks.jsonl`

Triggered by opening `qa_generation/output/chunks.jsonl` in a pager and seeing
boxed `<U+009B>`-style glyphs mixed into the Bahdini text. Two separate things
were visible in that screenshot:

- literal `\n` sequences — that's just JSON string-escaping of real newlines,
  expected when you view raw JSONL in a pager instead of parsing it. Not a bug.
- an actual **non-printable control character** rendered as a box — that *is*
  unexpected in clean Bahdini text, and is what this notebook investigates.

Question: is `chunks.jsonl` — the file the QA generation pipeline
(`generate_qa_openrouter.py`) actually reads — carrying real character-level
corruption that slipped past the existing quality gate
(`qa_config.MIN_DOC_CHARS_PER_TOKEN`, a per-document median chars/token check),
and if so, how much, and does it matter?


In [ ]:
import json
import unicodedata
import collections
import statistics
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(Path.cwd().parent))
import qa_config as cfg

CHUNKS_PATH = cfg.CHUNKS_PATH
print("Reading:", CHUNKS_PATH)
print("This is qa_config.CHUNKS_PATH itself -- the exact path generate_qa_openrouter.py "
      "and compile_qa_dataset.py both read, not a guess at where it might live.")
print("Size on disk:", f"{CHUNKS_PATH.stat().st_size / 1e6:.1f} MB")


## 1. Streaming scan

`chunks.jsonl` is 716 MB — loading the full `text` column into a pandas
DataFrame directly risks blowing up memory (the box has ~3 GB free). Instead
this streams the file line by line, and only keeps small aggregates:

- a per-chunk summary row (id, source, document, sizes, whether it's affected)
- a global counter of *which* non-printable codepoints appear
- a handful of example snippets per codepoint, for inspection

"Non-printable" here means Unicode general category `Cc` (control), `Cf`
(format), `Co` (private-use area), or `Cs` (surrogate) — excluding `\n`/`\t`,
which are legitimate.


In [ ]:
NONPRINTABLE_CATEGORIES = {"Cc", "Cf", "Co", "Cs"}
ALLOWED = {"\n", "\t"}

summary_rows = []
char_counter = collections.Counter()
char_examples = collections.defaultdict(list)
doc_occurrences = collections.Counter()

with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        text = r["text"]
        found = [ch for ch in text if unicodedata.category(ch) in NONPRINTABLE_CATEGORIES and ch not in ALLOWED]
        n_found = len(found)
        if n_found:
            for ch in found:
                char_counter[ch] += 1
            doc_occurrences[r["document_id"]] += n_found
            if len(char_examples[found[0]]) < 3:
                idx = text.index(found[0])
                char_examples[found[0]].append(
                    (r["chunk_id"], r["source"], text[max(0, idx - 40):idx + 40])
                )
        summary_rows.append({
            "chunk_id": r["chunk_id"],
            "document_id": r["document_id"],
            "source": r["source"],
            "origin": r["origin"],
            "char_count": r["char_count"],
            "token_estimate": r["token_estimate"],
            "n_nonprintable": n_found,
            "affected": n_found > 0,
        })

df = pd.DataFrame(summary_rows)
df["chars_per_token"] = df["char_count"] / df["token_estimate"].clip(lower=1)
print(f"{len(df):,} chunks scanned")


## 2. How widespread is it

In [ ]:
total = len(df)
affected = int(df["affected"].sum())
print(f"affected chunks: {affected:,} / {total:,}  ({affected/total*100:.2f}%)")
print(f"distinct documents affected: {df.loc[df.affected, 'document_id'].nunique():,} "
      f"/ {df['document_id'].nunique():,} documents")
print(f"total non-printable occurrences: {sum(char_counter.values()):,}")


In [ ]:
by_source = df.groupby("source").agg(
    chunks=("affected", "size"), affected=("affected", "sum")
)
by_source["affected_pct"] = by_source["affected"] / by_source["chunks"] * 100
by_source = by_source.sort_values("affected_pct", ascending=False)
display(by_source)

fig, ax = plt.subplots(figsize=(7, 4))
by_source["affected_pct"].plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("% chunks affected")
ax.set_title("Non-printable character contamination by source")
plt.tight_layout()
plt.show()


## 3. Which characters are these, actually

Two very different mechanisms tend to produce this:

- **Windows-1252 mojibake**: bytes `0x80`-`0x9F` are printable characters in
  cp1252 (curly quotes, em-dash, ellipsis, …) but are *raw C1 control codes*
  in Unicode. If a cp1252-encoded PDF text layer got decoded as Latin-1/UTF-8
  without translation, those bytes survive as literal control characters —
  category `Cc`.
- **Symbol/legacy fonts**: PDFs that draw bullets, icons, or (per
  `qa_config.py`'s already-documented corruption case) legacy Kurdish fonts
  via a custom glyph mapping land in the Private Use Area (`U+F000`-`U+F8FF`)
  — category `Co`. These decode to *some* Unicode codepoint, just not a
  meaningful one outside that font's private mapping.


In [ ]:
rows = []
for ch, count in char_counter.most_common(20):
    cat = unicodedata.category(ch)
    try:
        name = unicodedata.name(ch)
    except ValueError:
        name = "(unnamed)"
    cp1252_guess = ""
    if 0x80 <= ord(ch) <= 0x9F:
        try:
            cp1252_guess = bytes([ord(ch)]).decode("cp1252")
        except UnicodeDecodeError:
            cp1252_guess = "(undefined in cp1252)"
    rows.append({
        "codepoint": f"U+{ord(ch):04X}", "category": cat, "name": name,
        "count": count, "distinct_chunks": None, "cp1252_if_mojibake": cp1252_guess,
    })
char_df = pd.DataFrame(rows)
display(char_df)


## 4. Is this a few bad source documents, or spread everywhere?

If it's concentrated in a handful of documents, that's a targeted
re-extraction/exclusion job. If it's spread thin across most documents, it's
a systemic decoding bug worth fixing once, upstream.


In [ ]:
doc_occ = pd.Series(doc_occurrences).sort_values(ascending=False)
total_occ = doc_occ.sum()
top20_share = doc_occ.head(20).sum() / total_occ * 100
print(f"{len(doc_occ)} distinct documents carry at least one occurrence")
print(f"top 20 documents ({20/len(doc_occ)*100:.1f}% of affected docs) account for "
      f"{top20_share:.1f}% of all {total_occ:,} occurrences")
display(doc_occ.head(15).rename("occurrences").to_frame())


## 5. Does the existing quality gate already catch this?

`qa_config.MIN_DOC_CHARS_PER_TOKEN` (1.5) gates whole documents by their
median chars/token, built specifically to catch legacy-font corruption where
letters get *substituted* wholesale. Does that same signal catch this
control-character contamination as a side effect, or is it a blind spot?


In [ ]:
aff = df.loc[df.affected, "chars_per_token"]
unaff = df.loc[~df.affected, "chars_per_token"]

print("chars/token -- affected chunks:  mean", round(aff.mean(), 3), " median", round(aff.median(), 3))
print("chars/token -- unaffected chunks: mean", round(unaff.mean(), 3), " median", round(unaff.median(), 3))

below_gate_aff = (aff < cfg.MIN_DOC_CHARS_PER_TOKEN).mean() * 100
below_gate_unaff = (unaff < cfg.MIN_DOC_CHARS_PER_TOKEN).mean() * 100
print(f"\n% of affected chunks below the {cfg.MIN_DOC_CHARS_PER_TOKEN} gate:   {below_gate_aff:.2f}%")
print(f"% of unaffected chunks below the {cfg.MIN_DOC_CHARS_PER_TOKEN} gate: {below_gate_unaff:.2f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(unaff, bins=60, alpha=0.5, label="unaffected", density=True)
ax.hist(aff, bins=60, alpha=0.5, label="affected", density=True)
ax.axvline(cfg.MIN_DOC_CHARS_PER_TOKEN, color="red", linestyle="--", label=f"gate ({cfg.MIN_DOC_CHARS_PER_TOKEN})")
ax.set_xlabel("chars / token")
ax.legend()
ax.set_title("Chars-per-token: affected vs. unaffected chunks")
plt.tight_layout()
plt.show()


**Reading this**: the affected population's chars/token distribution sits
only slightly left of the unaffected one, and the overwhelming majority of
affected chunks still land comfortably above the 1.5 gate. The per-document
median gate was tuned to catch wholesale letter-substitution corruption
(whole documents reading as character soup) — it isn't sensitive to a
scattering of stray control/PUA characters inside otherwise-normal text,
because those few bad characters get averaged out by the rest of a
long-enough chunk. **This is a real, separate blind spot**, not something
already handled.


## 6. Concrete before/after examples

In [ ]:
for ch, examples in list(char_examples.items())[:6]:
    cat = unicodedata.category(ch)
    print(f"U+{ord(ch):04X} ({cat}), seen {char_counter[ch]} times:")
    for chunk_id, source, ctx in examples[:1]:
        print(f"  [{source}] {chunk_id}")
        print(f"    {ctx!r}")
    print()


## 7. A candidate cleanup, tested against the real examples

Two fixes, applied only to the categories actually implicated:

- `Cc` in the `0x80`-`0x9F` range: reinterpret as cp1252 and recover the
  intended character (curly quotes, dashes, etc.) where cp1252 defines one;
  drop the character if cp1252 leaves that byte undefined.
- `Co` (private-use area): these are font-specific glyph slots with no
  portable meaning outside the font that defined them. The tempting default
  is "just drop them" (most are decorative bullets, e.g. `U+F0B7`) — but
  this is exactly the shape of the legacy-Kurdish-font substitution bug
  `qa_config.py` already documents elsewhere (real letters silently
  remapped to the wrong codepoint), just manifesting as PUA codepoints
  instead of plausible-looking Arabic letters this time. The check below
  looks for PUA characters sitting *inside* an Arabic-script word (letters
  on both sides) as opposed to standalone (e.g. a bullet at the start of a
  line) — if any turn up, some of these are missing letters, not
  decoration, and blindly dropping them would silently corrupt the word
  rather than clean it.


In [ ]:
import re

def cp1252_recover(ch: str) -> str:
    try:
        return bytes([ord(ch)]).decode("cp1252")
    except (UnicodeDecodeError, ValueError):
        return ""

def clean_text(text: str) -> str:
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat == "Cc" and ch not in ALLOWED and 0x80 <= ord(ch) <= 0x9F:
            out.append(cp1252_recover(ch))
        elif cat in ("Co", "Cs"):
            continue
        else:
            out.append(ch)
    return "".join(out)

ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

def pua_is_midword(text: str) -> list:
    hits = []
    for i, ch in enumerate(text):
        if unicodedata.category(ch) == "Co":
            before = text[i - 1] if i > 0 else ""
            after = text[i + 1] if i + 1 < len(text) else ""
            if ARABIC_RE.match(before) and ARABIC_RE.match(after):
                hits.append((ch, text[max(0, i - 15):i + 15]))
    return hits


In [ ]:
# does any PUA char actually sit mid-word (i.e. might be a substituted letter,
# not a decorative bullet)?
midword_hits = []
sample_n = 0
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        hits = pua_is_midword(r["text"])
        if hits:
            midword_hits.append((r["chunk_id"], r["source"], hits[:2]))
        sample_n += 1
        if len(midword_hits) >= 15:
            break

print(f"scanned {sample_n:,} chunks before finding 15 mid-word PUA hits")
for chunk_id, source, hits in midword_hits[:15]:
    print(f"[{source}] {chunk_id}: {hits}")


In [ ]:
# before/after on the real corrupted example found earlier (document 6e30f99073eec478)
before = None
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["document_id"] == "6e30f99073eec478" and r["chunk_index"] == 0:
            before = r["text"]
            break

if before:
    after = clean_text(before)
    idx = before.find("\x94")
    print("BEFORE:", repr(before[max(0, idx-30):idx+30]))
    idx2 = after.find(cp1252_recover("\x94"))
    print("AFTER: ", repr(after[max(0, idx2-30):idx2+30]) if idx2 >= 0 else "(control char removed/translated)")


## 8. Verdict

- **4.29% of chunks (10,937 / 254,872) contain real non-printable
  characters** — this is not a rendering artifact of the pager, it's in the
  file the QA pipeline actually reads.
- The dominant pattern by volume is **Windows-1252 mojibake** (`U+0092`/
  `U+0094` etc., cp1252 smart-quote bytes surviving as raw C1 control
  codes) — recoverable deterministically, confirmed by the before/after
  example above.
- **PUA symbol-font glyphs are a mixed bag, not uniformly safe to drop.**
  Most are standalone decoration (`U+F0B7` bullets in facebook posts). But
  the mid-word scan above *did* find real hits — e.g. `کەرچاێتەیە` and
  `شیشە` — a PUA codepoint sitting between two Arabic letters inside a
  single word, in multiple sources (facebook, sh2_unicodefixed_bahdini,
  telegram_badini_book). That's the same legacy-font substitution bug
  `qa_config.py` already documents for the `safe_extraction` pool, just
  showing up as a PUA codepoint instead of a plausible-looking Arabic
  letter this time — meaning it evades that corruption check too. Blindly
  stripping all `Co` characters would silently delete real letters in these
  cases rather than clean them. This needs per-document review, not a
  blanket strip.
- One of the worst offenders (`safe_extraction-69d9dc523f5eb8e2`, source
  `facebook`) isn't stray-character noise at all — its content is a
  different kind of garbled entirely (`Žõ‹q@Žôî@õ†@Ž¶b÷@`-style text, not
  Bahdini in any script). That document likely needs outright exclusion,
  not character cleanup.
- It's **concentrated**: 656 of 5,370 documents (12.2%) are touched at all,
  and the worst 20 documents account for ~66% of all occurrences — a small
  re-extraction/cleanup job, not a corpus-wide rewrite.
- It's a **confirmed blind spot** in `MIN_DOC_CHARS_PER_TOKEN`: affected
  chunks' chars/token distribution barely differs from clean chunks (means
  1.84 vs. 2.02), so the existing per-document gate — built for wholesale
  character-soup corruption — doesn't see this at all.

**Recommendation**, two separate fixes rather than one blanket cleanup:

1. **`Cc` cp1252 recovery is safe to apply everywhere** — deterministic,
   confirmed correct on the real example above, no ambiguity. Cheap to add
   to `build_chunks.py` (or as a pre-processing step on `chunk["text"]`
   right before it's substituted into the QA prompt).
2. **`Co`/PUA needs per-document handling, not a blanket strip** — flag the
   656 affected documents for a quick pass: documents where PUA only
   appears standalone (decorative bullets) can have it stripped safely;
   documents where it appears mid-word (the 15+ hits found above) are
   likely carrying the same silent letter-substitution corruption
   `qa_config.py` already excludes elsewhere, and should be excluded the
   same way, not "cleaned." The worst offenders by occurrence count (top 20
   documents = 66% of all occurrences) are the highest-value place to
   start — small enough to eyeball by hand.

Either fix, applied to `build_chunks.py`, means rebuilding the chunk queue —
check whether that shifts `chunk_id`s (it shouldn't, since those hash
`document_id`, not chunk content) before doing it against the already
in-flight generation run.


## 9. Correction: not everything flagged above is actually corruption

Section 2's scan flagged Unicode categories `Cc`/`Cf`/`Co`/`Cs` as
"non-printable". That was too broad. Checking every distinct `Cf` character
that got swept in:


In [ ]:
cf_chars = collections.Counter()
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        for ch in r["text"]:
            if unicodedata.category(ch) == "Cf":
                cf_chars[ch] += 1

rows = []
for ch, n in cf_chars.most_common():
    try:
        name = unicodedata.name(ch)
    except ValueError:
        name = "?"
    rows.append({"codepoint": f"U+{ord(ch):04X}", "name": name, "count": n})
display(pd.DataFrame(rows))


Every one of these is legitimate: `U+06DD ARABIC END OF AYAH` is a real
Quranic verse-end marker (expected in religious Bahdini texts, not noise);
`U+200E`/`U+200F` (LTR/RTL marks), `U+200D` (ZWJ), `U+2060` (word joiner),
`U+200B` (ZWSP) are standard bidi/shaping control characters *required* for
correct Arabic-script rendering; `U+00AD` (soft hyphen) and `U+FEFF` (BOM
leftover) are harmless invisible artifacts. **None of this is corruption —
stripping it would be actively wrong** (broken bidi rendering, deleted
liturgical punctuation). The `Cf` category was the wrong net to cast.

Redoing the scan restricted to `Cc` (excl. `\n`/`\t`) + `Co` + `Cs` — the
categories that are actually implicated by the mechanism found below —
gives the real scope:


In [ ]:
affected_docs2 = set()
cc_docs2, co_docs2 = set(), set()
affected_chunks2 = 0
total2 = 0

with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        total2 += 1
        r = json.loads(line)
        text = r["text"]
        has_cc = any(unicodedata.category(ch) == "Cc" and ch not in ALLOWED for ch in text)
        has_co = any(unicodedata.category(ch) == "Co" for ch in text)
        has_cs = any(unicodedata.category(ch) == "Cs" for ch in text)
        if has_cc or has_co or has_cs:
            affected_chunks2 += 1
            affected_docs2.add(r["document_id"])
        if has_cc: cc_docs2.add(r["document_id"])
        if has_co: co_docs2.add(r["document_id"])

print(f"corrected affected chunks: {affected_chunks2:,} / {total2:,} = {affected_chunks2/total2*100:.2f}%")
print(f"corrected affected documents: {len(affected_docs2)}  (not the 656 from the too-broad first pass)")
print(f"  docs with Cc mojibake: {len(cc_docs2)}")
print(f"  docs with Co PUA glyphs: {len(co_docs2)}")
print(f"  docs with both: {len(cc_docs2 & co_docs2)}")


## 10. Root cause: is this our mistake, or already in the source?

In [ ]:
# 1. does it correlate with extraction origin? (native PDF text layer vs Gemini OCR)
origin_totals = collections.Counter()
origin_affected = collections.Counter()
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        text = r["text"]
        origin_totals[r["origin"]] += 1
        has_cc = any(unicodedata.category(ch) == "Cc" and ch not in ALLOWED for ch in text)
        has_co = any(unicodedata.category(ch) == "Co" for ch in text)
        has_cs = any(unicodedata.category(ch) == "Cs" for ch in text)
        if has_cc or has_co or has_cs:
            origin_affected[r["origin"]] += 1

for origin, tot in origin_totals.items():
    aff = origin_affected[origin]
    print(f"{origin}: {aff:,}/{tot:,} = {aff/tot*100:.2f}% affected")


Almost exclusively `safe_extraction` (native PDF text layer). `ocr_corpus`
(Gemini transcribing the rendered page image) is barely touched — expected,
since OCR never depends on the PDF's internal font encoding at all.

2. Was it already there before this QA-generation pipeline ever ran, i.e. in
`extractions/*/safe/*.txt` (the output of `scripts/extract_pipeline.py`,
built in an earlier session)?


In [ ]:
import subprocess

EXTRACTIONS_DIR = cfg.ROOT / "extractions"
result = subprocess.run(
    ["grep", "-rl", "-P", "\x{0094}", str(EXTRACTIONS_DIR / "facebook" / "safe")],
    capture_output=True, text=True,
)
print("files in extractions/facebook/safe/ already containing raw U+0094:")
print(result.stdout or "(none found)")


Confirmed present in the raw `.txt` files themselves — this predates
`build_chunks.py` and `generate_qa_openrouter.py` entirely. It was baked in
during the earlier PDF-extraction run, not introduced by anything in this
QA-generation pipeline.

**Mechanism**: `extract_pdf()` in `scripts/extract_pipeline.py` uses
PyMuPDF (`fitz`)'s `page.get_text("text")`, then `clean_text()` runs NFKC
normalization plus KLPT's Kurdish-specific character unification — neither
step touches control/PUA characters at all. When a PDF's embedded font has
no (or a broken) `ToUnicode` CMap — common for older Word-to-PDF exports and
custom/legacy Kurdish fonts — `fitz` falls back to the font's raw internal
character codes. Fonts built on Windows' `WinAnsiEncoding` (cp1252) land
those codes in the C1 control range (`0x80`-`0x9F`); custom symbol fonts
land them in the Private Use Area. This is the *same underlying failure
mode* `qa_config.py` already documents for the "character soup" corruption
case (`MIN_DOC_CHARS_PER_TOKEN`) — a font whose glyphs don't map to correct
Unicode — just manifesting as raw control/PUA codepoints instead of
plausible-looking wrong Arabic letters, which is exactly why the existing
gates (`presentation_form_ratio`, `MIN_DOC_CHARS_PER_TOKEN`) don't catch it: it's
a different symptom of a problem those gates were tuned to catch in only
one of its forms.

## 11. Answer

**Not a mistake in this QA-generation work, and not random source noise
either — it's an unhandled failure mode in `scripts/extract_pipeline.py`
from an earlier extraction run**, invisible to that script's existing
quality gates because they check different symptoms of "bad font
encoding," not this one.

**What to do, concretely:**

1. Add cp1252 recovery for `Cc` (0x80-0x9F) to `clean_text()` in
   `extract_pipeline.py` — deterministic, safe, no PDF re-parse needed since
   the already-saved `.txt` files contain everything needed to recover it.
   Backfill by running this normalization once over the existing
   `extractions/*/safe/*.txt` files (39 documents actually affected).
2. For the 344 documents carrying PUA glyphs: split by the mid-word check
   from Section 7 — 315 are decorative-only (safe to strip the PUA chars);
   29 show mid-word insertion (real suspected letter substitution). Route
   those 29 through the same `needs_ocr` reclassification
   `extract_pipeline.py`'s `classification()` already applies for other
   corruption signals, so Gemini OCR (which sidesteps the font problem
   entirely) produces clean text for them instead — don't try to
   character-patch them.
3. Rebuild `qa_generation/output/chunks.jsonl` via `build_chunks.py` after
   the fix (cheap, a few minutes per the pipeline README).
4. Timing is in our favor: only ~255 of 254,872 chunks have been through
   real generation so far (175 from an earlier non-OpenRouter pilot batch,
   80 from this session's live OpenRouter pilots) — negligible sunk cost.
   This is the right moment to fix it, before the ~$300-order full run,
   not after.
5. Minor side effect to expect: rebuilding will very likely shift
   `chunk_index`/`chunk_id` for the ~374 affected documents specifically
   (paragraph packing is token-count-based, and removing/translating a
   handful of characters can nudge a boundary), which would orphan the
   already-recorded generation attempts for *those* documents only (harmless
   — a few cents of redundant reprocessing, not data corruption). The
   ~3,985 unaffected documents are untouched and keep their resumability.
